In [0]:
# Databricks notebook source
# ══════════════════════════════════════
# GOLD — KPIs Dados Externos
# Squad 3 — Arquitetura Medalhao
# KPI 7L : Enriquecimento IBGE
# KPI 10 : Flag venda_em_feriado
# Frequencia: toda segunda-feira as 5h
# ══════════════════════════════════════

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# constantes do notebook

SILVER_ITENS_PATH = f"{SILVER_BASE_PATH}physical_itens_venda_caixa"
SILVER_LOJAS_PATH = f"{SILVER_BASE_PATH}physical_lojas"

# caminhos dados externos
IBGE_PATH      = f"{RAW_ROOT_PATH}reference-data/ibge_municipios"
FERIADOS_PATH  = f"{RAW_ROOT_PATH}reference-data/feriados_nacionais"

# caminhos Gold
GOLD_KPI7L_PATH  = f"{GOLD_BASE_PATH}kpi_lojas_enriquecimento_ibge"
GOLD_KPI7L_TABLE = f"{TARGET_SCHEMA}.gold_kpi_lojas_enriquecimento_ibge"

GOLD_KPI10_PATH  = f"{GOLD_BASE_PATH}kpi_vendas_feriado_loja"
GOLD_KPI10_TABLE = f"{TARGET_SCHEMA}.gold_kpi_vendas_feriado_loja"

GOLD_WRITE_MODE = "overwrite"

print("Constantes configuradas:")
print(f"   SILVER_ITENS_PATH : {SILVER_ITENS_PATH}")
print(f"   SILVER_LOJAS_PATH : {SILVER_LOJAS_PATH}")
print(f"   IBGE_PATH         : {IBGE_PATH}")
print(f"   FERIADOS_PATH     : {FERIADOS_PATH}")

In [0]:
# configuracoes do ADLS

adls_options = get_adls_options()
print("Opcoes ADLS configuradas.")

In [0]:
# ler Silver itens e lojas

df_itens = read_delta(
    spark        = spark,
    path         = SILVER_ITENS_PATH,
    adls_options = adls_options
)
print(f"Silver itens : {df_itens.count():,} linhas")

df_lojas = read_delta(
    spark        = spark,
    path         = SILVER_LOJAS_PATH,
    adls_options = adls_options
)
print(f"Silver lojas : {df_lojas.count():,} linhas")

display(df_lojas.limit(5))

In [0]:
# converter tipos

from pyspark.sql.functions import (
    col, count, sum as spark_sum,
    round as spark_round, when,
    current_timestamp, to_date,
    upper, trim, lit
)
from pyspark.sql.types import IntegerType, DoubleType

df_itens = (
    df_itens
    .withColumn("valor_total_item",
        col("valor_total_item").cast(DoubleType()))
    .withColumn("id_loja",
        col("id_loja").cast(IntegerType()))
    .withColumn("ano",
        col("ano").cast(IntegerType()))
    .withColumn("mes",
        col("mes").cast(IntegerType()))
)

df_lojas = (
    df_lojas
    .withColumn("id_loja",
        col("id_loja").cast(IntegerType()))
)

print("Tipos convertidos com sucesso!")

In [0]:
# ══════════════════════════════════════
# PARTE 1 — IBGE
# Buscar e salvar municipios
# ══════════════════════════════════════

import requests
import pandas as pd

def buscar_municipios_ibge():
    """
    Busca dados de municipios brasileiros
    via API IBGE (gratuita).
    URL: servicodados.ibge.gov.br
    """
    url      = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
    response = requests.get(url, timeout=30)

    if response.status_code != 200:
        raise Exception(
            f"Erro ao buscar municipios IBGE: "
            f"status={response.status_code}"
        )

    municipios = response.json()

    df = pd.DataFrame([{
        "id_municipio"      : str(m["id"]),
        "nome_municipio"    : m["nome"].upper().strip(),
        "uf"                : (m.get("microrregiao") or {}).get("mesorregiao", {}).get("UF", {}).get("sigla") or 
                              (m.get("regiao-imediata") or {}).get("regiao-intermediaria", {}).get("UF", {}).get("sigla"),
        "nome_uf"           : (m.get("microrregiao") or {}).get("mesorregiao", {}).get("UF", {}).get("nome") or 
                              (m.get("regiao-imediata") or {}).get("regiao-intermediaria", {}).get("UF", {}).get("nome"),
        "nome_regiao"       : (m.get("microrregiao") or {}).get("mesorregiao", {}).get("UF", {}).get("regiao", {}).get("nome") or 
                              (m.get("regiao-imediata") or {}).get("regiao-intermediaria", {}).get("UF", {}).get("regiao", {}).get("nome"),
        "nome_mesorregiao"  : (m.get("microrregiao") or {}).get("mesorregiao", {}).get("nome"),
        "nome_microrregiao" : (m.get("microrregiao") or {}).get("nome"),
    } for m in municipios])

    return df

print("Buscando municipios via API IBGE...")
df_municipios_pd = buscar_municipios_ibge()

print(f"Total de municipios: {len(df_municipios_pd):,}")
print(df_municipios_pd.head(5))

In [0]:
# salvar municipios IBGE no Data Lake

df_municipios_spark = (
    spark.createDataFrame(df_municipios_pd)
    .withColumn("dt_carga", current_timestamp())
)

write_delta(
    df           = df_municipios_spark,
    path         = IBGE_PATH,
    mode         = "overwrite",
    adls_options = adls_options
)

print(f"Municipios IBGE salvos em: {IBGE_PATH}")
display(df_municipios_spark.limit(10))

In [0]:
# ler IBGE gravado

df_ibge = read_delta(
    spark        = spark,
    path         = IBGE_PATH,
    adls_options = adls_options
)

print(f"Total municipios IBGE: {df_ibge.count():,}")
display(df_ibge.limit(5))

In [0]:
# fazer JOIN lojas x IBGE

df_lojas_enriquecida = (
    df_lojas
    .join(
        df_ibge.select(
            "id_municipio",
            col("nome_municipio").alias("cidade_ibge"),
            col("uf").alias("uf_ibge"),
            "nome_regiao",
            "nome_mesorregiao",
            "nome_microrregiao",
        ),
        (upper(trim(col("cidade_loja"))) == col("cidade_ibge")) &
        (col("estado_loja") == col("uf_ibge")),
        how="left"
    )
    .drop("cidade_ibge", "uf_ibge")
    .withColumn("gold_processed_at", current_timestamp())
)

# verificar qualidade do JOIN
total_lojas    = df_lojas.count()
com_match_ibge = df_lojas_enriquecida.filter(
    col("id_municipio").isNotNull()
).count()
sem_match_ibge = total_lojas - com_match_ibge

print("=" * 55)
print("QUALIDADE JOIN IBGE")
print("=" * 55)
print(f"   Total lojas          : {total_lojas:,}")
print(f"   Com match IBGE       : {com_match_ibge:,}")
print(f"   Sem match IBGE       : {sem_match_ibge:,}")
print(f"   Taxa de match        : {com_match_ibge/total_lojas*100:.1f}%")

if sem_match_ibge > 0:
    print(f"\nLojas sem match IBGE:")
    display(
        df_lojas_enriquecida
        .filter(col("id_municipio").isNull())
        .select(
            "id_loja",
            "nome_loja",
            "cidade_loja",
            "estado_loja"
        )
    )

display(df_lojas_enriquecida.limit(10))

In [0]:
# gravar KPI 7L — Lojas enriquecidas com IBGE

write_delta(
    df           = df_lojas_enriquecida,
    path         = GOLD_KPI7L_PATH,
    mode         = GOLD_WRITE_MODE,
    adls_options = adls_options
)

write_sql_table(
    df         = df_lojas_enriquecida,
    table_name = GOLD_KPI7L_TABLE,
    mode       = GOLD_WRITE_MODE
)

print("KPI 7L gravado com sucesso!")

In [0]:
# validar KPI 7L

df_kpi7l_saved = read_delta(
    spark        = spark,
    path         = GOLD_KPI7L_PATH,
    adls_options = adls_options
)

compare_row_counts(
    source_df = df_lojas_enriquecida,
    target_df = df_kpi7l_saved,
    label     = "KPI 7L memoria x Delta gravado"
)

In [0]:
# COMMAND ----------

# ══════════════════════════════════════
# PARTE 2 — FERIADOS
# Buscar e salvar feriados nacionais
# ══════════════════════════════════════

def buscar_feriados_brasil(anos):
    """
    Busca feriados nacionais do Brasil
    via BrasilAPI (gratuita).
    URL: brasilapi.com.br/api/feriados/v1/{ano}
    """
    df_list = []

    for ano in anos:
        url      = f"https://brasilapi.com.br/api/feriados/v1/{ano}"
        response = requests.get(url, timeout=10)

        if response.status_code != 200:
            print(
                f"Atencao: erro ao buscar feriados "
                f"de {ano}: status={response.status_code}"
            )
            continue

        feriados = response.json()
        df_ano   = pd.DataFrame(feriados)
        df_ano["ano"] = ano
        df_list.append(df_ano)
        print(f"   Feriados {ano}: {len(df_ano)} encontrados")

    if not df_list:
        raise Exception(
            "Nenhum feriado encontrado. "
            "Verifique a conexao com a BrasilAPI."
        )

    return pd.concat(df_list, ignore_index=True)

# buscar anos presentes nos dados
anos_disponiveis = [
    row.ano for row in
    df_itens
    .select("ano")
    .distinct()
    .orderBy("ano")
    .collect()
]

print(f"Anos nos dados: {anos_disponiveis}")
print("Buscando feriados via BrasilAPI...")

df_feriados_pd = buscar_feriados_brasil(anos_disponiveis)

print(f"\nTotal de feriados: {len(df_feriados_pd):,}")
print(df_feriados_pd)

In [0]:
# salvar feriados no Data Lake

df_feriados_spark = (
    spark.createDataFrame(df_feriados_pd)
    .withColumn("dt_carga", current_timestamp())
)

write_delta(
    df           = df_feriados_spark,
    path         = FERIADOS_PATH,
    mode         = "overwrite",
    partition_by = ["ano"],
    adls_options = adls_options
)

print(f"Feriados salvos em: {FERIADOS_PATH}")
display(df_feriados_spark.orderBy("date").limit(20))

In [0]:
# ler feriados gravados

df_feriados = read_delta(
    spark        = spark,
    path         = FERIADOS_PATH,
    adls_options = adls_options
)

print(f"Total feriados: {df_feriados.count():,}")
display(df_feriados.orderBy("date"))

In [0]:
# preparar referencia de feriados para JOIN

df_feriados_ref = df_feriados.select(
    to_date(col("date")).alias("data_feriado"),
    col("name").alias("nome_feriado"),
    col("type").alias("tipo_feriado"),
)

print("Schema feriados referencia:")
df_feriados_ref.printSchema()
display(df_feriados_ref.limit(10))

In [0]:
# adicionar flag venda_em_feriado nos itens

df_itens_com_flag = (
    df_itens
    .withColumn("data_venda_date",
        to_date(col("dt_venda")))
    .join(
        df_feriados_ref,
        col("data_venda_date") == col("data_feriado"),
        how="left"
    )
    .withColumn(
        "venda_em_feriado",
        when(col("nome_feriado").isNotNull(), True)
        .otherwise(False)
    )
    .drop("data_venda_date", "data_feriado")
)

# verificar distribuicao
total_itens   = df_itens_com_flag.count()
em_feriado    = df_itens_com_flag.filter(
    col("venda_em_feriado") == True
).count()
fora_feriado  = total_itens - em_feriado

print("=" * 55)
print("DISTRIBUICAO VENDAS EM FERIADO")
print("=" * 55)
print(f"   Total de itens        : {total_itens:,}")
print(f"   Vendas em feriado     : {em_feriado:,}")
print(f"   Vendas fora feriado   : {fora_feriado:,}")
print(f"   % em feriado          : {em_feriado/total_itens*100:.2f}%")

display(
    df_itens_com_flag
    .filter(col("venda_em_feriado") == True)
    .select(
        "id_item_venda",
        "id_loja",
        "dt_venda",
        "nome_feriado",
        "tipo_feriado",
        "venda_em_feriado",
        "valor_total_item"
    )
    .limit(10)
)

In [0]:
# calcular KPI 10 — Vendas em feriado por loja

df_kpi10 = (
    df_itens_com_flag
    .groupBy(
        "id_loja",
        "ano",
        "mes",
        "venda_em_feriado",
        "nome_feriado",
        "tipo_feriado"
    )
    .agg(
        count("id_item_venda").alias("total_itens"),
        spark_round(
            spark_sum("valor_total_item"), 2
        ).alias("receita_total"),
        count(col("id_transacao")).alias("total_transacoes"),
    )
    .join(
        df_lojas.select(
            "id_loja", "nome_loja",
            "cidade_loja", "estado_loja"
        ),
        on="id_loja",
        how="left"
    )
    .withColumn("gold_processed_at", current_timestamp())
)

total_kpi10 = df_kpi10.count()
print(f"KPI 10 — Vendas feriado/loja: {total_kpi10:,} linhas")

display(
    df_kpi10
    .filter(col("venda_em_feriado") == True)
    .orderBy(col("receita_total").desc())
    .limit(10)
)

In [0]:
# analise — feriados com maior receita

print("Top feriados por receita total:")
display(
    df_kpi10
    .filter(col("venda_em_feriado") == True)
    .groupBy("nome_feriado", "tipo_feriado")
    .agg(
        spark_round(
            spark_sum("receita_total"), 2
        ).alias("receita_total_feriado"),
        spark_sum("total_transacoes").alias("total_transacoes"),
    )
    .orderBy(col("receita_total_feriado").desc())
)

In [0]:
# gravar KPI 10 — Vendas em feriado

write_delta(
    df           = df_kpi10,
    path         = GOLD_KPI10_PATH,
    mode         = GOLD_WRITE_MODE,
    partition_by = ["ano", "mes"],
    adls_options = adls_options
)

write_sql_table(
    df         = df_kpi10,
    table_name = GOLD_KPI10_TABLE,
    mode       = GOLD_WRITE_MODE
)

print("KPI 10 gravado com sucesso!")

In [0]:
# validar KPI 10

df_kpi10_saved = read_delta(
    spark        = spark,
    path         = GOLD_KPI10_PATH,
    adls_options = adls_options
)

compare_row_counts(
    source_df = df_kpi10,
    target_df = df_kpi10_saved,
    label     = "KPI 10 memoria x Delta gravado"
)